# Clase 4 — Redes convolucionales (CNN) con Keras

## Objetivos de aprendizaje
- Entender por qué una red convolucional (CNN) es más adecuada que una red
  densa para trabajar con imágenes.
- Construir una CNN con una capa `Conv2D` + `MaxPooling2D` y comparar sus
  resultados con la red densa de la clase anterior.
- Construir una segunda CNN más profunda, con dos rondas de convolución y
  pooling.
- Entender qué es el `batch_size` y cómo afecta al entrenamiento.

## Qué vamos a construir
Dos arquitecturas de red convolucional entrenadas sobre el mismo dataset
MNIST de la clase anterior, para poder comparar directamente "red densa
vs. CNN" y "CNN simple vs. CNN más profunda".

**Nota sobre tiempos de ejecución:** el material original de IBM entrena
estos modelos con `epochs=10`. En este notebook usamos **3 épocas** para
que la clase corra en un tiempo razonable en un Colab con CPU (unos
minutos en vez de decenas de minutos). Si quieres reproducir los
resultados de precisión reportados en el manual (que rondan el 98-99%),
puedes subir `epochs` a 10 cuando practiques por tu cuenta, especialmente
si activas una GPU en Colab (`Entorno de ejecución → Cambiar tipo de
entorno de ejecución → GPU`).

Keras y TensorFlow ya vienen preinstalados en Colab: no hace falta ningún
`!pip install` para esta clase.

## Repaso rápido (capítulo 7 del manual)

Una CNN combina tres tipos de capas:
- **`Conv2D`**: desliza filtros pequeños sobre la imagen para detectar
  patrones locales (bordes, curvas), reutilizando el mismo filtro en toda
  la imagen.
- **`MaxPooling2D`**: reduce el tamaño de los mapas de características
  quedándose con el valor máximo de cada región pequeña.
- **`Flatten`**: aplana el resultado antes de pasarlo a capas `Dense`
  normales, que hacen la clasificación final con softmax.

A diferencia de la red densa de la clase anterior, aquí las imágenes
**no** se aplanan de entrada: se mantienen en formato
`(alto, ancho, canales)`.

In [ ]:
import keras
from keras.models import Sequential
from keras.layers import Dense, Input
from keras.layers import Conv2D        # capas convolucionales
from keras.layers import MaxPooling2D  # capas de pooling (agrupamiento)
from keras.layers import Flatten       # aplanar antes de las capas densas
from keras.utils import to_categorical
from keras.datasets import mnist

# MNIST viene incluido en Keras, igual que en la clase anterior.
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Reshape a formato imagen [muestras][alto][ancho][canales]. A diferencia
# de la red densa, aquí NO aplanamos: mantenemos la estructura espacial.
X_train = X_train.reshape(X_train.shape[0], 28, 28, 1).astype('float32')
X_test = X_test.reshape(X_test.shape[0], 28, 28, 1).astype('float32')

# Normalizamos los píxeles y codificamos las etiquetas en one-hot, igual
# que en la clase anterior.
X_train = X_train / 255
X_test = X_test / 255
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)
num_classes = y_test.shape[1]

print("X_train:", X_train.shape)
print("Número de clases:", num_classes)

## CNN 1 — Un solo set de convolución + pooling

16 filtros de 5x5 en la capa convolucional, seguidos de un pooling 2x2, y
luego las mismas capas densas de la red de clasificación de la clase
anterior.

In [ ]:
def convolutional_model():
    model = Sequential()
    model.add(Input(shape=(28, 28, 1)))
    model.add(Conv2D(16, (5, 5), strides=(1, 1), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))

    model.add(Flatten())
    model.add(Dense(100, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

model = convolutional_model()
model.summary()

In [ ]:
# NOTA: el material original de IBM usa epochs=10. Aquí usamos 3 épocas
# para que la clase corra en pocos minutos en un Colab con CPU; sube a 10
# si quieres reproducir la precisión exacta reportada en el manual.
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=3, batch_size=200, verbose=2)

scores = model.evaluate(X_test, y_test, verbose=0)
print("Precisión: {:.4f} \n Error: {:.4f}".format(scores[1], 100 - scores[1] * 100))

## CNN 2 — Dos sets de convolución + pooling

Ahora añadimos una segunda ronda de `Conv2D` + `MaxPooling2D` antes de
aplanar, para que la red pueda combinar los patrones simples detectados en
la primera ronda en características más abstractas.

In [ ]:
def convolutional_model_v2():
    model = Sequential()
    model.add(Input(shape=(28, 28, 1)))
    model.add(Conv2D(16, (5, 5), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))

    # segunda ronda: extrae características más abstractas a partir de
    # las que ya detectó la primera ronda
    model.add(Conv2D(8, (2, 2), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))

    model.add(Flatten())
    model.add(Dense(100, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

model_v2 = convolutional_model_v2()
model_v2.summary()

In [ ]:
# Igual que antes: 3 épocas en vez de las 10 del material original, para
# tiempos de ejecución razonables en clase.
model_v2.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=3, batch_size=200, verbose=2)

scores_v2 = model_v2.evaluate(X_test, y_test, verbose=0)
print("Precisión: {:.4f} \n Error: {:.4f}".format(scores_v2[1], 100 - scores_v2[1] * 100))

## Resumen de la clase

- Una CNN procesa imágenes en su formato original `(alto, ancho, canales)`,
  sin aplanar de entrada, a diferencia de una red densa.
- `Conv2D` detecta patrones locales reutilizando el mismo filtro en toda
  la imagen; `MaxPooling2D` reduce dimensionalidad quedándose con los
  valores máximos; `Flatten` aplana antes de las capas densas finales.
- Apilar más de una ronda de convolución + pooling permite aprender
  características más abstractas.
- `batch_size` controla cuántas muestras ve la red antes de actualizar sus
  pesos una vez; en la práctica de esta clase vas a experimentar
  cambiándolo.
- Con solo 3 épocas ya se pueden ver resultados sólidos (arriba del 95% de
  precisión); con las 10 épocas del material original la precisión sube
  todavía más, típicamente por encima del 98%.

**Sigue con:**
- Capítulo del manual: [`07_cnn.md`](../manual/07_cnn.md)
- Notebook de práctica de esta clase: [`clase04_practica_cnn.ipynb`](../notebooks_practica/clase04_practica_cnn.ipynb)